<a href="https://colab.research.google.com/github/mcjkurz/qhchina/blob/main/tutorials/Intro_to_Python_for_Chinese_Humanities_Part_5_OCR.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install opencv-python
!apt-get install tesseract-ocr-chi-sim
!pip install pytesseract
import random

!wget https://github.com/mcjkurz/qhchina/raw/refs/heads/main/tutorials/data/白樺_苦戀.jpg

In [ ]:
import cv2
import pytesseract
from PIL import Image
from google.colab.patches import cv2_imshow # trick to show the image

# Load the image using OpenCV
image_path = '白樺_苦戀.jpg'
image = cv2.imread(image_path)
cv2_imshow(image)

In [ ]:
# Perform OCR on the image
recognized_text = pytesseract.image_to_string(image, lang='chi_sim')  # 'chi_sim' for Simplified Chinese

# Print the recognized text
print("Recognized Text:")
print(recognized_text)

In [ ]:
#https://becominghuman.ai/how-to-automatically-deskew-straighten-a-text-image-using-opencv-a0c30aed83df
import numpy as np

def getSkewAngle(cvImage) -> float:
    # Prep image, copy, convert to gray scale, blur, and threshold
    newImage = cvImage.copy()
    gray = cv2.cvtColor(newImage, cv2.COLOR_BGR2GRAY)
    blur = cv2.GaussianBlur(gray, (9, 9), 0)
    thresh = cv2.threshold(blur, 0, 255, cv2.THRESH_BINARY_INV + cv2.THRESH_OTSU)[1]

    # Apply dilate to merge text into meaningful lines/paragraphs.
    # Use larger kernel on X axis to merge characters into single line, cancelling out any spaces.
    # But use smaller kernel on Y axis to separate between different blocks of text
    kernel = cv2.getStructuringElement(cv2.MORPH_RECT, (30, 5))
    dilate = cv2.dilate(thresh, kernel, iterations=2)

    # Find all contours
    contours, hierarchy = cv2.findContours(dilate, cv2.RETR_LIST, cv2.CHAIN_APPROX_SIMPLE)
    contours = sorted(contours, key = cv2.contourArea, reverse = True)
    for c in contours:
        rect = cv2.boundingRect(c)
        x,y,w,h = rect
        cv2.rectangle(newImage,(x,y),(x+w,y+h),(0,255,0),2)

    # Find largest contour and surround in min area box
    largestContour = contours[0]
    print (len(contours))
    minAreaRect = cv2.minAreaRect(largestContour)
    cv2.imwrite("boxes.jpg", newImage)
    # Determine the angle. Convert it to the value that was originally used to obtain skewed image
    angle = minAreaRect[-1]
    if angle < -45:
        angle = 90 + angle
    return -1.0 * angle
# Rotate the image around its center
def rotateImage(cvImage, angle: float):
    newImage = cvImage.copy()
    (h, w) = newImage.shape[:2]
    center = (w // 2, h // 2)
    M = cv2.getRotationMatrix2D(center, angle, 1.0)
    newImage = cv2.warpAffine(newImage, M, (w, h), flags=cv2.INTER_CUBIC, borderMode=cv2.BORDER_REPLICATE)
    return newImage

# Deskew image
def deskew(cvImage):
    angle = getSkewAngle(cvImage)
    return rotateImage(cvImage, -1.0 * angle)

In [ ]:
rotated_image = deskew(image)
cv2.imwrite("rotated_image.jpg", rotated_image)
cv2_imshow(rotated_image)

# Perform OCR on the image
recognized_text = pytesseract.image_to_string(rotated_image, lang='chi_sim')  # 'chi_sim' for Simplified Chinese

# Print the recognized text
print("Recognized Text:")
print(recognized_text)

In [ ]:
gray_image = cv2.cvtColor(rotated_image, cv2.COLOR_BGR2GRAY)
cv2_imshow(gray_image)

# Perform OCR on the image
recognized_text = pytesseract.image_to_string(gray_image, lang='chi_sim')  # 'chi_sim' for Simplified Chinese

# Print the recognized text
print("Recognized Text:")
print(recognized_text)

In [ ]:
def noise_removal(image):
    import numpy as np
    kernel = np.ones((1, 1), np.uint8)
    image = cv2.dilate(image, kernel, iterations=1)
    kernel = np.ones((1, 1), np.uint8)
    image = cv2.erode(image, kernel, iterations=1)
    image = cv2.morphologyEx(image, cv2.MORPH_CLOSE, kernel)
    #image = cv2.medianBlur(image, 1)
    return (image)

In [ ]:
denoised_image = noise_removal(gray_image)
denoised_image = cv2.medianBlur(gray_image, 1)
denoised_image = cv2.fastNlMeansDenoising(gray_image, None, 20, 7, 21)
cv2.imwrite("denoised_image.jpg", denoised_image)
cv2_imshow(denoised_image)

# Perform OCR on the image
recognized_text = pytesseract.image_to_string(denoised_image, lang='chi_sim')  # 'chi_sim' for Simplified Chinese

# Print the recognized text
print("Recognized Text:")
print(recognized_text)

In [ ]:
def remove_borders(image):
    contours, heiarchy = cv2.findContours(image, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    cntsSorted = sorted(contours, key=lambda x:cv2.contourArea(x))
    cnt = cntsSorted[-1]
    x, y, w, h = cv2.boundingRect(cnt)
    crop = image[y:y+h, x:x+w]
    return (crop)

In [ ]:
noborders_image = remove_borders(denoised_image)
cv2.imwrite("noborders_image.jpg", noborders_image)
cv2_imshow(noborders_image)

# Perform OCR on the image
recognized_text = pytesseract.image_to_string(noborders_image, lang='chi_sim')  # 'chi_sim' for Simplified Chinese

# Print the recognized text
print("Recognized Text:")
print(recognized_text)

In [ ]:
# alternatively we can just manually remove some part of the image

noborders_image = denoised_image[50:denoised_image.shape[0]-50,50:denoised_image.shape[1]-50]
cv2.imwrite("noborders_image.jpg", noborders_image)
cv2_imshow(noborders_image)

# Perform OCR on the image
recognized_text = pytesseract.image_to_string(noborders_image, lang='chi_sim')  # 'chi_sim' for Simplified Chinese

# Print the recognized text
print("Recognized Text:")
print(recognized_text)

In [ ]:
thresh, bw_image = cv2.threshold(noborders_image, None, 255, cv2.THRESH_BINARY | cv2.THRESH_OTSU)
cv2.imwrite("bw_image.jpg", bw_image)
cv2_imshow(bw_image)
recognized_text = pytesseract.image_to_string(bw_image, lang='chi_sim')  # 'chi_sim' for Simplified Chinese

# Print the recognized text
print("Recognized Text:")
print(recognized_text)

In [ ]:
preprocessed_text = [line for line in recognized_text.split("\n") if len(line.strip()) > 0]
preprocessed_text = "".join(preprocessed_text)
print(preprocessed_text)